In [55]:
from arch import arch_model
import pandas as pd
import numpy as np
import statsmodels.api as sm

stock = r"C:\Users\Dominique\Desktop\QF603\proj\all_tickers_time_series.hf5"
options = r"C:\Users\Dominique\Desktop\QF603\proj\all_options_data.h5"

In [48]:
stock_data = pd.read_hdf(stock, key = "AAPL")[["date", "prc"]].drop_duplicates()
stock_data.set_index("date", inplace = True)
stock_returns = stock_data.pct_change()*100
stock_returns = stock_returns.dropna()

dff = pd.read_csv("DFF.csv", index_col=0)
dff.index = pd.to_datetime(dff.index, dayfirst= True)

brent = pd.read_csv("brent.csv", index_col=0)
brent.index = pd.to_datetime(brent.index, dayfirst=True)
brent = brent.replace(".", np.nan).fillna(method="ffill")
brent_returns = brent.astype(float).pct_change()*100
brent_returns = brent_returns.dropna()

# inflation = pd.read_excel("inflation.xlsx", index_col=0)[["Inflation rate"]].loc["2000-01-01":"2023-12-01"]

C:\Users\Dominique\AppData\Local\Temp\ipykernel_28708\3114774816.py:11: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  brent = brent.replace(".", np.nan).fillna(method="ffill")


In [50]:
print(len(stock_returns))
print(len(dff))
print(len(brent_returns))
# print(len(inflation))

6036
8764
6258


In [51]:
dates_sr = set(stock_returns.index)
dates_dff = set(dff.index) #includes everyday even weekends and PH
dates_brent = set(brent_returns.index)

common_dates = dates_sr & dates_dff & dates_brent
diff_dates = (dates_sr | dates_dff | dates_brent) - common_dates

print(len(common_dates))
print(len(diff_dates))

6035
2729


In [52]:
stock_returns = stock_returns[stock_returns.index.isin(common_dates)]
dff = dff[dff.index.isin(common_dates)]
brent_returns = brent_returns[brent_returns.index.isin(common_dates)]

In [58]:
combined_df = pd.concat([dff,brent_returns], axis = 1)
combined_df.columns = ["dff","brent"]

X = combined_df
Y = stock_returns
ols_model = sm.OLS(Y,X).fit()
residuals = ols_model.resid

In [59]:
garch_model = arch_model(residuals, vol='Garch', p=1, q=1)
fit = garch_model.fit()
fit.summary()

Iteration:      1,   Func. Count:      6,   Neg. LLF: 25796.697300670046
Iteration:      2,   Func. Count:     15,   Neg. LLF: 23738.405017030786
Iteration:      3,   Func. Count:     22,   Neg. LLF: 20837.404760512058
Iteration:      4,   Func. Count:     29,   Neg. LLF: 14930.04574877656
Iteration:      5,   Func. Count:     35,   Neg. LLF: 14916.977077813113
Iteration:      6,   Func. Count:     41,   Neg. LLF: 15340.42702289725
Iteration:      7,   Func. Count:     48,   Neg. LLF: 14932.673803333568
Iteration:      8,   Func. Count:     54,   Neg. LLF: 15356.07525476706
Iteration:      9,   Func. Count:     60,   Neg. LLF: 14990.102787421483
Iteration:     10,   Func. Count:     66,   Neg. LLF: 14898.301667089636
Iteration:     11,   Func. Count:     71,   Neg. LLF: 14897.996573263394
Iteration:     12,   Func. Count:     76,   Neg. LLF: 14897.972399709668
Iteration:     13,   Func. Count:     81,   Neg. LLF: 14897.971554654214
Iteration:     14,   Func. Count:     86,   Neg. LLF: 

c:\Users\Dominique\anaconda3\lib\site-packages\arch\univariate\base.py:1897: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if isinstance(val[pos], np.float64):
c:\Users\Dominique\anaconda3\lib\site-packages\arch\univariate\base.py:1898: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  converted = format_float_fixed(val[pos], *formats[i])


<class 'statsmodels.iolib.summary.Summary'>
"""
                     Constant Mean - GARCH Model Results                      
==============================================================================
Dep. Variable:                   None   R-squared:                       0.000
Mean Model:             Constant Mean   Adj. R-squared:                  0.000
Vol Model:                      GARCH   Log-Likelihood:               -14898.0
Distribution:                  Normal   AIC:                           29803.9
Method:            Maximum Likelihood   BIC:                           29830.8
                                        No. Observations:                 6035
Date:                Fri, Oct 11 2024   Df Residuals:                     6034
Time:                        16:42:05   Df Model:                            1
                               Mean Model                               
========================================================================
                 coef    std err          t      P>|t|  95.0% Conf. Int.
------------------------------------------------------------------------
mu            -0.0297  9.322e-02     -0.319      0.750 [ -0.212,  0.153]
                             Volatility Model                             
==========================================================================
                 coef    std err          t      P>|t|    95.0% Conf. Int.
--------------------------------------------------------------------------
omega          0.3183      0.361      0.883      0.377   [ -0.388,  1.025]
alpha[1]       0.0524  2.582e-02      2.031  4.228e-02 [1.827e-03,  0.103]
beta[1]        0.9259  2.451e-02     37.771      0.000   [  0.878,  0.974]
==========================================================================

Covariance estimator: robust
"""

In [62]:
garch_x = arch_model(Y, vol="Garch", p=1, q=1, x = X)
fit2 = garch_x.fit()
fit2.summary()

Iteration:      1,   Func. Count:      6,   Neg. LLF: 25591.092214485914
Iteration:      2,   Func. Count:     15,   Neg. LLF: 23634.86489919064
Iteration:      3,   Func. Count:     22,   Neg. LLF: 21084.898680783735
Iteration:      4,   Func. Count:     29,   Neg. LLF: 14941.109326089416
Iteration:      5,   Func. Count:     35,   Neg. LLF: 14937.645052693259
Iteration:      6,   Func. Count:     41,   Neg. LLF: 14938.391986243332
Iteration:      7,   Func. Count:     47,   Neg. LLF: 15143.957181005258
Iteration:      8,   Func. Count:     53,   Neg. LLF: 15112.964657055194
Iteration:      9,   Func. Count:     59,   Neg. LLF: 14940.550230963714
Iteration:     10,   Func. Count:     65,   Neg. LLF: 14913.15741778388
Iteration:     11,   Func. Count:     71,   Neg. LLF: 14912.7671876568
Iteration:     12,   Func. Count:     77,   Neg. LLF: 14912.201126552614
Iteration:     13,   Func. Count:     82,   Neg. LLF: 14912.200901192416
Iteration:     14,   Func. Count:     87,   Neg. LLF: 1

c:\Users\Dominique\anaconda3\lib\site-packages\arch\univariate\base.py:1897: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if isinstance(val[pos], np.float64):
c:\Users\Dominique\anaconda3\lib\site-packages\arch\univariate\base.py:1898: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  converted = format_float_fixed(val[pos], *formats[i])


<class 'statsmodels.iolib.summary.Summary'>
"""
                     Constant Mean - GARCH Model Results                      
==============================================================================
Dep. Variable:                    prc   R-squared:                       0.000
Mean Model:             Constant Mean   Adj. R-squared:                  0.000
Vol Model:                      GARCH   Log-Likelihood:               -14912.2
Distribution:                  Normal   AIC:                           29832.4
Method:            Maximum Likelihood   BIC:                           29859.2
                                        No. Observations:                 6035
Date:                Fri, Oct 11 2024   Df Residuals:                     6034
Time:                        16:44:28   Df Model:                            1
                                Mean Model                               
=========================================================================
                  coef    std err          t      P>|t|  95.0% Conf. Int.
-------------------------------------------------------------------------
mu         -5.0763e-03  9.236e-02 -5.496e-02      0.956 [ -0.186,  0.176]
                              Volatility Model                              
============================================================================
                 coef    std err          t      P>|t|      95.0% Conf. Int.
----------------------------------------------------------------------------
omega          0.2989      0.334      0.895      0.371     [ -0.355,  0.953]
alpha[1]       0.0496  2.508e-02      1.978  4.788e-02 [4.631e-04,9.876e-02]
beta[1]        0.9301  2.150e-02     43.254      0.000     [  0.888,  0.972]
============================================================================

Covariance estimator: robust
"""

numbers are... almost exactly the same???